# Flu Shot Learning - notebook 11 (AUC-aware rank blend)

This experiment builds on notebook 9 with target-specific probability and percentile/rank blending. Because the competition metric is ROC AUC, the blend is selected from out-of-fold predictions and can use rank-normalized model outputs to reduce differences in probability calibration. No test labels are used for model or blend selection.

Expected local OOF reference: notebook 9 achieved 0.86933 mean ROC AUC. Run this notebook end-to-end and compare the printed OOF score before uploading its submission.

In [1]:
from pathlib import Path
import importlib.util
import sys

missing = []
if importlib.util.find_spec('catboost') is None:
    missing.append('catboost')
if importlib.util.find_spec('lightgbm') is None:
    missing.append('lightgbm')
if importlib.util.find_spec('iterstrat') is None:
    missing.append('iterative-stratification')
if missing:
    !"{sys.executable}" -m pip install -q {' '.join(missing)}

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 2026
N_SPLITS = 5
TARGETS = ['h1n1_vaccine', 'seasonal_vaccine']
pd.set_option('display.max_columns', 100)
print('Ready')

Ready


In [2]:
def find_data_dir():
    candidates = [
        Path('/kaggle/input/datasets/barshamishra19/flu-shot-learning-dataset'),
       
    ]
    for candidate in candidates:
        if (candidate / 'training_set_features.csv').exists():
            return candidate
    raise FileNotFoundError('Set DATA_DIR to the folder containing the competition CSV files.')

DATA_DIR = find_data_dir()
train_features = pd.read_csv(DATA_DIR / 'training_set_features.csv')
train_labels = pd.read_csv(DATA_DIR / 'training_set_labels.csv')
test_features = pd.read_csv(DATA_DIR / 'test_set_features.csv')
assert train_features['respondent_id'].is_unique
assert test_features['respondent_id'].is_unique
assert train_features['respondent_id'].equals(train_labels['respondent_id'])
assert set(train_features['respondent_id']).isdisjoint(test_features['respondent_id'])
y = train_labels[TARGETS].astype('int8')
print(f'Using: {DATA_DIR.resolve()}')
print(f'Train: {train_features.shape}; test: {test_features.shape}')
display(y.mean().rename('positive_rate').to_frame())

Using: /kaggle/input/datasets/barshamishra19/flu-shot-learning-dataset
Train: (26707, 36); test: (26708, 36)


,positive_rate
h1n1_vaccine,0.212454
seasonal_vaccine,0.465608


In [3]:
def engineer_features(frame):
    result = frame.drop(columns=['respondent_id'], errors='ignore').copy()
    result['missing_count'] = result.isna().sum(axis=1)
    result['missing_fraction'] = result.isna().mean(axis=1)
    behavior = [c for c in result if c.startswith('behavioral_')]
    opinions = [c for c in result if c.startswith('opinion_')]
    risk = [c for c in opinions if '_risk' in c]
    result['behavior_score'] = result[behavior].sum(axis=1, min_count=1)
    result['behavior_available'] = result[behavior].notna().sum(axis=1)
    result['opinion_score'] = result[opinions].mean(axis=1)
    result['opinion_risk_score'] = result[risk].mean(axis=1)
    result['doctor_recommendation_count'] = result[['doctor_recc_h1n1', 'doctor_recc_seasonal']].sum(axis=1, min_count=1)
    result['health_access_score'] = result[['health_insurance', 'health_worker']].sum(axis=1, min_count=1)
    result['recommendation_gap'] = result['doctor_recc_seasonal'] - result['doctor_recc_h1n1']
    result['opinion_effective_gap'] = result['opinion_seas_vacc_effective'] - result['opinion_h1n1_vacc_effective']
    result['opinion_risk_gap'] = result['opinion_seas_risk'] - result['opinion_h1n1_risk']
    result['opinion_sick_gap'] = result['opinion_seas_sick_from_vacc'] - result['opinion_h1n1_sick_from_vacc']
    result['opinion_effective_mean'] = result[['opinion_seas_vacc_effective', 'opinion_h1n1_vacc_effective']].mean(axis=1)
    result['opinion_risk_mean'] = result[['opinion_seas_risk', 'opinion_h1n1_risk']].mean(axis=1)
    result['behavior_opinion_alignment'] = result['behavior_score'] * result['opinion_score']
    result['doctor_opinion_alignment'] = result['doctor_recommendation_count'] * result['opinion_score']
    result['health_worker_insured'] = result['health_worker'] * result['health_insurance']
    ordinal_maps = {
        'age_group': {'18 - 34 Years': 1, '35 - 44 Years': 2, '45 - 54 Years': 3, '55 - 64 Years': 4, '65+ Years': 5},
        'education': {'< 12 Years': 1, '12 Years': 2, 'Some College': 3, 'College Graduate': 4},
        'income_poverty': {'Below Poverty': 1, '<= $75,000, Above Poverty': 2, '> $75,000': 3},
    }
    for column, mapping in ordinal_maps.items():
        result[column + '_ordinal'] = result[column].map(mapping)
    return result

X = engineer_features(train_features)
X_test = engineer_features(test_features)
categorical_columns = X.select_dtypes(include=['object', 'category']).columns.tolist()
for column in categorical_columns:
    X[column] = X[column].fillna('__MISSING__').astype(str)
    X_test[column] = X_test[column].fillna('__MISSING__').astype(str)
cat_feature_indices = [X.columns.get_loc(c) for c in categorical_columns]
X_lgb = X.copy()
X_test_lgb = X_test.copy()
for column in categorical_columns:
    levels = pd.concat([X_lgb[column], X_test_lgb[column]], ignore_index=True).unique()
    dtype = pd.api.types.CategoricalDtype(categories=levels)
    X_lgb[column] = X_lgb[column].astype(dtype)
    X_test_lgb[column] = X_test_lgb[column].astype(dtype)
stratify_key = y[TARGETS].astype(str).agg('_'.join, axis=1)
folds = list(MultilabelStratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE).split(X, y))
print(f'Engineered features: {X.shape[1]}; categorical: {len(categorical_columns)}')

Engineered features: 55; categorical: 12


In [4]:
def make_catboost(target, seed):
    settings = {
        'h1n1_vaccine': {'depth': 7, 'iterations': 2300, 'learning_rate': 0.028, 'l2_leaf_reg': 9.0},
        'seasonal_vaccine': {'depth': 6, 'iterations': 2300, 'learning_rate': 0.028, 'l2_leaf_reg': 10.0},
    }[target]
    return CatBoostClassifier(**settings, loss_function='Logloss', eval_metric='AUC', random_seed=seed, verbose=False, allow_writing_files=False, thread_count=-1)

def make_lightgbm(seed, target):
    return LGBMClassifier(objective='binary', n_estimators=3000, learning_rate=0.018, num_leaves=24 if target == 'h1n1_vaccine' else 31, max_depth=-1, min_child_samples=35, subsample=0.9, colsample_bytree=0.9, reg_alpha=0.15, reg_lambda=1.5, random_state=seed, n_jobs=-1, verbosity=-1)

numeric_columns = X.select_dtypes(exclude=['object', 'category']).columns.tolist()
linear_preprocessor = ColumnTransformer([
    ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median', add_indicator=True)), ('scale', StandardScaler(with_mean=False))]), numeric_columns),
    ('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('one_hot', OneHotEncoder(handle_unknown='ignore', sparse_output=True))]), categorical_columns),
], sparse_threshold=1.0)

def make_linear(target, c_value):
    return Pipeline([('preprocessor', linear_preprocessor), ('classifier', LogisticRegression(C=c_value, solver='saga', penalty='l2', max_iter=3000, random_state=RANDOM_STATE, n_jobs=-1))])

In [5]:
model_names = ['catboost', 'lightgbm', 'linear_lo', 'linear_hi']
oof = {name: pd.DataFrame(index=X.index, columns=TARGETS, dtype=float) for name in model_names}
test_pred = {name: pd.DataFrame(0.0, index=X_test.index, columns=TARGETS) for name in model_names}
for fold_number, (train_index, valid_index) in enumerate(folds, start=1):
    print(f'Fold {fold_number}/{N_SPLITS}')
    for target in TARGETS:
        cat = make_catboost(target, RANDOM_STATE + fold_number)
        cat.fit(X.iloc[train_index], y.iloc[train_index][target], cat_features=cat_feature_indices, eval_set=(X.iloc[valid_index], y.iloc[valid_index][target]), use_best_model=True, early_stopping_rounds=170)
        oof['catboost'].loc[valid_index, target] = cat.predict_proba(X.iloc[valid_index])[:, 1]
        test_pred['catboost'][target] += cat.predict_proba(X_test)[:, 1] / N_SPLITS

        lgb = make_lightgbm(RANDOM_STATE + 100 + fold_number, target)
        lgb.fit(X_lgb.iloc[train_index], y.iloc[train_index][target], eval_set=[(X_lgb.iloc[valid_index], y.iloc[valid_index][target])], eval_metric='auc', categorical_feature=categorical_columns, callbacks=[early_stopping(170, verbose=False), log_evaluation(0)])
        oof['lightgbm'].loc[valid_index, target] = lgb.predict_proba(X_lgb.iloc[valid_index])[:, 1]
        test_pred['lightgbm'][target] += lgb.predict_proba(X_test_lgb)[:, 1] / N_SPLITS

        for name, c_value in [('linear_lo', 0.12), ('linear_hi', 0.55)]:
            linear = make_linear(target, c_value)
            linear.fit(X.iloc[train_index], y.iloc[train_index][target])
            oof[name].loc[valid_index, target] = linear.predict_proba(X.iloc[valid_index])[:, 1]
            test_pred[name][target] += linear.predict_proba(X_test)[:, 1] / N_SPLITS

model_scores = pd.DataFrame({name: [roc_auc_score(y[target], oof[name][target]) for target in TARGETS] for name in model_names}, index=TARGETS)
model_scores.loc['mean'] = model_scores.mean(axis=0)
display(model_scores)

Fold 1/5
Fold 2/5
Fold 3/5
Fold 4/5
Fold 5/5


,catboost,lightgbm,linear_lo,linear_hi
h1n1_vaccine,0.871788,0.870946,0.862790,0.862504
seasonal_vaccine,0.864998,0.864712,0.855067,0.855006
mean,0.868393,0.867829,0.858928,0.858755


In [6]:
# Compare probability and rank-normalized blends using OOF predictions only.
def rank_normalize(values):
    return pd.Series(values).rank(method='average', pct=True).to_numpy()

def target_blend_score(weights, target, mode='probability'):
    if mode == 'rank':
        blended = sum(weights[name] * rank_normalize(oof[name][target]) for name in model_names)
    else:
        blended = sum(weights[name] * oof[name][target].to_numpy() for name in model_names)
    return roc_auc_score(y[target], blended)

weight_grid = np.arange(0.0, 1.01, 0.1)
best_weights = {}
best_modes = {}
best_target_scores = {}
for target in TARGETS:
    candidates = []
    for cat_weight in weight_grid:
        for lgb_weight in weight_grid:
            for linear_lo_weight in weight_grid:
                linear_hi_weight = round(1.0 - cat_weight - lgb_weight - linear_lo_weight, 10)
                if linear_hi_weight < 0:
                    continue
                weights = {'catboost': cat_weight, 'lightgbm': lgb_weight, 'linear_lo': linear_lo_weight, 'linear_hi': linear_hi_weight}
                for mode in ('probability', 'rank'):
                    candidates.append((target_blend_score(weights, target, mode), mode, weights))
    score, mode, weights = max(candidates, key=lambda item: item[0])
    best_weights[target] = weights
    best_modes[target] = mode
    best_target_scores[target] = score
    print(target, mode, weights, 'OOF AUC:', round(score, 6))
print('Mean OOF ROC AUC:', round(float(np.mean(list(best_target_scores.values()))), 6))


h1n1_vaccine probability {'catboost': np.float64(0.5), 'lightgbm': np.float64(0.4), 'linear_lo': np.float64(0.1), 'linear_hi': np.float64(-0.0)} OOF AUC: 0.872654
seasonal_vaccine rank {'catboost': np.float64(0.5), 'lightgbm': np.float64(0.5), 'linear_lo': np.float64(0.0), 'linear_hi': np.float64(0.0)} OOF AUC: 0.86602
Mean OOF ROC AUC: 0.869337


In [7]:
final_predictions = pd.DataFrame(index=X_test.index, columns=TARGETS, dtype=float)
for target in TARGETS:
    if best_modes[target] == 'rank':
        candidates = {name: rank_normalize(test_pred[name][target]) for name in model_names}
    else:
        candidates = {name: test_pred[name][target].to_numpy() for name in model_names}
    final_predictions[target] = sum(best_weights[target][name] * candidates[name] for name in model_names)

submission = pd.DataFrame({'respondent_id': test_features['respondent_id'], 'h1n1_vaccine': np.clip(final_predictions['h1n1_vaccine'], 0.0, 1.0), 'seasonal_vaccine': np.clip(final_predictions['seasonal_vaccine'], 0.0, 1.0)})
assert list(submission.columns) == ['respondent_id', 'h1n1_vaccine', 'seasonal_vaccine']
assert submission['respondent_id'].equals(test_features['respondent_id'])
assert submission[TARGETS].notna().all().all()
assert ((submission[TARGETS] >= 0) & (submission[TARGETS] <= 1)).all().all()
output_dir = Path.cwd() / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)
submission_path = output_dir / 'submission_notebook11_auc_rank_blend.csv'
oof_path = output_dir / 'oof_notebook11_auc_rank_blend.csv'
submission.to_csv(submission_path, index=False)
oof_frame = oof['catboost'].rename(columns=lambda c: f'catboost_{c}')
for name in model_names[1:]:
    oof_frame = oof_frame.join(oof[name].rename(columns=lambda c, n=name: f'{n}_{c}'))
oof_frame.insert(0, 'respondent_id', train_features['respondent_id'])
oof_frame.to_csv(oof_path, index=False)
print(f'Saved submission: {submission_path.resolve()}')
print(f'Saved OOF predictions: {oof_path.resolve()}')
display(submission.head())


Saved submission: /kaggle/working/outputs/submission_notebook11_auc_rank_blend.csv
Saved OOF predictions: /kaggle/working/outputs/oof_notebook11_auc_rank_blend.csv


,respondent_id,h1n1_vaccine,seasonal_vaccine
0,26707,0.115660,0.306556
1,26708,0.028142,0.019863
2,26709,0.183076,0.675228
3,26710,0.632301,0.861989
4,26711,0.289815,0.549798
